# Етап 1.3 — LinkedIn: OAuth, профіль, тестовий пост

### Ручні кроки (один раз)
1. Мати або створити **LinkedIn Page**: застосунок без неї не створити, підійде тестова сторінка.
2. https://www.linkedin.com/developers/apps → **Create app**, прив'язати до сторінки.
3. Вкладка **Products** → додати **Sign In with LinkedIn using OpenID Connect** і **Share on LinkedIn**.
4. Вкладка **Auth** → *Authorized redirect URLs* → `http://localhost:8765/callback`.
5. Скопіювати *Client ID* і *Client Secret* у файл `.secrets/linkedin.env` у корені репозиторію:
```
LINKEDIN_CLIENT_ID=...
LINKEDIN_CLIENT_SECRET=...
```
`.secrets/` у `.gitignore`.

Доступні без схвалення LinkedIn тільки `openid`, `profile`, `email` (читати ім'я / фото / email) і `w_member_social` (публікувати пости). Редагувати профіль через API не можна.

In [ ]:
import json, secrets, time, webbrowser
from http.server import BaseHTTPRequestHandler, HTTPServer
from pathlib import Path
from urllib.parse import urlencode, urlparse, parse_qs
import httpx

ROOT = Path.cwd().parent
SECRETS = ROOT / ".secrets"
TOKEN_FILE = SECRETS / "linkedin_token.json"
REDIRECT_URI = "http://localhost:8765/callback"
SCOPES = "openid profile email w_member_social"
LINKEDIN_VERSION = "202609"   # формат YYYYMM; перевір актуальну версію в документації Posts API

def load_env(path):
    return dict(line.split("=", 1) for line in path.read_text().splitlines() if "=" in line and not line.startswith("#"))

env = load_env(SECRETS / "linkedin.env") if (SECRETS / "linkedin.env").exists() else {}
print("Client ID знайдено" if env.get("LINKEDIN_CLIENT_ID") else "Немає .secrets/linkedin.env — див. кроки вище")

### OAuth: отримати access token

Відкриється браузер, ти підтверджуєш доступ, LinkedIn редиректить на `localhost:8765`, і клітинка ловить `code` та обмінює його на токен. Токен живе близько 60 днів.

In [ ]:
def oauth_login():
    state = secrets.token_urlsafe(16)
    url = "https://www.linkedin.com/oauth/v2/authorization?" + urlencode({
        "response_type": "code", "client_id": env["LINKEDIN_CLIENT_ID"],
        "redirect_uri": REDIRECT_URI, "state": state, "scope": SCOPES,
    })
    result = {}

    class Handler(BaseHTTPRequestHandler):
        def do_GET(self):
            q = parse_qs(urlparse(self.path).query)
            result.update({k: v[0] for k, v in q.items()})
            self.send_response(200); self.end_headers()
            self.wfile.write("Готово, можна закрити вкладку.".encode())
        def log_message(self, *a):
            pass

    server = HTTPServer(("localhost", 8765), Handler)
    webbrowser.open(url)
    server.handle_request()          # waits for exactly one redirect
    server.server_close()
    assert result.get("state") == state, f"state mismatch / error: {result}"
    r = httpx.post("https://www.linkedin.com/oauth/v2/accessToken", data={
        "grant_type": "authorization_code", "code": result["code"], "redirect_uri": REDIRECT_URI,
        "client_id": env["LINKEDIN_CLIENT_ID"], "client_secret": env["LINKEDIN_CLIENT_SECRET"],
    })
    r.raise_for_status()
    tok = r.json() | {"obtained_at": int(time.time())}
    SECRETS.mkdir(exist_ok=True)
    TOKEN_FILE.write_text(json.dumps(tok, indent=2))
    print("Токен збережено, діє днів:", tok.get("expires_in", 0) // 86400, "| scope:", tok.get("scope"))

# oauth_login()

### Профіль: `/v2/userinfo`

In [ ]:
def token():
    return json.loads(TOKEN_FILE.read_text())["access_token"]

def userinfo():
    r = httpx.get("https://api.linkedin.com/v2/userinfo", headers={"Authorization": f"Bearer {token()}"})
    r.raise_for_status()
    return r.json()

# me = userinfo(); me

### Тестовий пост (Posts API)

`POST https://api.linkedin.com/rest/posts`, автор `urn:li:person:<sub>`. За замовчуванням `POST_FOR_REAL = False`: клітинка тільки показує тіло запиту.

In [ ]:
POST_FOR_REAL = False

def publish_post(text, visibility="CONNECTIONS"):
    me = userinfo()
    body = {
        "author": f"urn:li:person:{me['sub']}",
        "commentary": text,
        "visibility": visibility,               # PUBLIC | CONNECTIONS
        "distribution": {"feedDistribution": "MAIN_FEED", "targetEntities": [], "thirdPartyDistributionChannels": []},
        "lifecycleState": "PUBLISHED",
        "isReshareDisabledByAuthor": False,
    }
    if not POST_FOR_REAL:
        return {"dry_run": True, "body": body}
    r = httpx.post("https://api.linkedin.com/rest/posts", json=body, headers={
        "Authorization": f"Bearer {token()}", "LinkedIn-Version": LINKEDIN_VERSION,
        "X-Restli-Protocol-Version": "2.0.0", "Content-Type": "application/json",
    })
    r.raise_for_status()
    return {"post_id": r.headers.get("x-restli-id")}

# publish_post("Тестовий пост з нашого навчального проєкту.")